# late-search-budget -- Astra's candidate against eat-rest-v1, 32 paired seeds

Candidate: `external/candidates/eat-rest-v1-late-search-budget` (from `eat-rest-v1-late-search-budget.zip`). It is v1 plus one
block: a late-game gate (never before t=1300; needs a 20 s average of <=6 agents and <180 energy, plus either failed young
replacements after t=1400 or food-and-predator pressure after t=1600). Once active, a hungry young agent (energy < 80, age < 60)
that is searching with no fruit known and no predator within 180 moves slower, on a 20 s energy budget. Before the gate fires
the agent is v1, action for action. See `README.md` / `mechanism.md` in the candidate folder.

Both agents play seeds 11000-11031 through `external/candidates/endgame_probe.py`; the last block of cell 2 is the paired
per-seed difference. 32 seeds gives a standard error of about 45, so only a difference beyond roughly +-90 means anything.
Look at `late_activated` first: the gate cannot fire in a game that ends before t=1300-1400, and v1's mean on these seeds
is about 1200. Resumable; baseline games already stored in `logs/eg2/base` are reused. Nothing is written to `results/`.

**Cluster setup:** same as `parameter-tuning.ipynb` (`.env` with `GITHUB_TOKEN=<token>`).

In [ ]:
import os

CLONE_DIR = "/home/jovyan/Nordic-AI-cup-2026"
if os.path.isdir(os.path.join(CLONE_DIR, ".git")):
    print(f"{CLONE_DIR} already cloned - skipping (use `git pull` there to update)")
else:
    # GitHub token is read from a git-ignored .env (GITHUB_TOKEN=...) in the kernel's cwd, or from the environment
    if os.path.isfile(".env"):
        for line in open(".env"):
            key, sep, value = line.strip().partition("=")
            if sep and not key.startswith("#"):
                os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))
    TOKEN = os.environ.get("GITHUB_TOKEN")
    if not TOKEN:
        raise RuntimeError(f"GITHUB_TOKEN not set - create {os.path.abspath('.env')} containing GITHUB_TOKEN=<token>")
    !git clone https://{TOKEN}@github.com/sjoeen/Nordic-AI-cup-2026.git {CLONE_DIR}

In [ ]:
import glob
import os
import subprocess
import sys

os.chdir(CLONE_DIR)
!git fetch origin challenge-1V2
!git checkout challenge-1V2
!git pull origin challenge-1V2

# Must run from survival-simulator/ so `src`, `agents`, `training` import.
if os.path.basename(os.getcwd()) != "survival-simulator":
    candidates = sorted({os.path.realpath(p) for p in glob.glob(os.path.join(os.getcwd(), "**", "survival-simulator"), recursive=True)
                         if os.path.isfile(os.path.join(p, "requirements.txt"))})
    if len(candidates) != 1:
        raise RuntimeError(f"cwd is {os.getcwd()}; found {len(candidates)} survival-simulator checkouts {candidates} - %cd into the right one")
    os.chdir(candidates[0])

print("cwd:", os.getcwd())
sys.path.insert(0, os.getcwd())
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
!{sys.executable} -m pip install -r requirements.txt -r requirements-dev.txt

## 1. Baseline: eat-rest-v1 on 32 seeds (the 16 already stored are reused, ~4 min)

In [ ]:
SEEDS = 32
!{sys.executable} -u external/candidates/endgame_probe.py --out logs/eg2/base --seeds {SEEDS} --brief 2>&1 | grep --line-buffered -v "pkg_resources\|pygame"

## 2. Astra's candidate on the same seeds: full report, then the paired comparison (~8 min)

In [ ]:
CANDIDATE = "eat-rest-v1-late-search-budget"
!{sys.executable} -u external/candidates/endgame_probe.py --out logs/eg2/{CANDIDATE} --candidate {CANDIDATE} --seeds {SEEDS} --compare logs/eg2/base 2>&1 | grep --line-buffered -v "pkg_resources\|pygame"

## Paired comparison only (no new games)

In [ ]:
!{sys.executable} -u external/candidates/endgame_probe.py --out logs/eg2/{CANDIDATE} --candidate {CANDIDATE} --seeds {SEEDS} --compare logs/eg2/base --brief 2>&1 | grep --line-buffered -v "pkg_resources\|pygame"